### **GDPR Right to Full Erasure**

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.governance")

TARGET = "ACC-100027"   # the customer who exercised their right to erasure

# readings are keyed by meter_id, so capture this customer's meters FIRST
meter_ids = [r.meter_id for r in spark.sql(
    f"SELECT meter_id FROM workspace.silver.meters WHERE account_id = '{TARGET}'").collect()]
meters_sql = "(" + ",".join(f"'{m}'" for m in meter_ids) + ")" if meter_ids else "('__none__')"
print("meters to erase:", meter_ids)

In [0]:
targets = {
    "workspace.bronze.customers":           f"account_id = '{TARGET}'",
    "workspace.bronze.meters":              f"account_id = '{TARGET}'",
    "workspace.bronze.invoices":            f"account_id = '{TARGET}'",
    "workspace.bronze.payments":            f"account_id = '{TARGET}'",
    "workspace.bronze.readings":            f"meter_id IN {meters_sql}",
    "workspace.bronze.readings_drift":      f"meter_id IN {meters_sql}",
    "workspace.silver.customers":           f"account_id = '{TARGET}'",
    "workspace.silver.meters":              f"account_id = '{TARGET}'",
    "workspace.silver.invoices":            f"account_id = '{TARGET}'",
    "workspace.silver.payments":            f"account_id = '{TARGET}'",
    "workspace.silver.readings":            f"meter_id IN {meters_sql}",
    "workspace.silver.readings_quarantine": f"meter_id IN {meters_sql}",
    "workspace.gold.dim_customer":          f"account_id = '{TARGET}'",
    "workspace.gold.dim_meter":             f"account_id = '{TARGET}'",
    "workspace.gold.fact_consumption":      f"account_id = '{TARGET}'",
    "workspace.gold.fact_billing":          f"account_id = '{TARGET}'",
}

for table, predicate in targets.items():
    before = spark.table(table).count()
    spark.sql(f"DELETE FROM {table} WHERE {predicate}")
    print(f"{table:42s} removed {before - spark.table(table).count()}")

In [0]:
# Serverless locks the 0-hour override, and that's fine — GDPR gives you a window, not an instant.
# The rows are already deleted from every current table (Cell 2). VACUUM clears the historical
# files; with the default ~7-day retention, the just-deleted data ages out and is purged.
for table in targets:
    spark.sql(f"VACUUM {table}")     # default retention; allowed on serverless
    print("vacuumed:", table)